# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
"""
Lane: Refresh / Content Opportunity Scoring. Confirming this week's lane lock per the card.

Before writing any rule, I checked the two signals it would lean on — both are signals
behind real FlyRank flags from the session, not invented ones.

"""

In [ ]:
"""
### Signal check 1 — staleness, behind the refresh flags

Bucket: freshness_tier (days since last update). Outcome: is_declining_label rate, with n.
"""

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

order = ["0-30", "31-90", "91-180", "181+"]
staleness_table = df.groupby("freshness_tier")["is_declining_label"].agg(["mean", "count"]).reindex(order)
staleness_table.columns = ["decline_rate", "n"]
print(staleness_table)

In [ ]:
"""
Verdict: MIXED.

Decline rate by freshness tier: 0-30=0.511 (n=20,480), 31-90=0.589 (n=175), 91-180=0.611
(n=9,171), 181+=0.471 (n=174). If staleness truly drove decline, I'd expect a clean upward
climb. Instead it rises through the first three tiers, then drops at the stalest tier — and
the two tiers that break the pattern (31-90, 181+) both have tiny samples (~175 rows each,
under 1% of the data), so I don't trust them either way. This is a real, explained negative:
staleness alone is not a reliable driver of decline in this snapshot, so my rule below does
NOT lean its main weight on it.
"""

In [ ]:
"""
### Signal check 2 — CTR vs. position, behind the CTR-fix logic

Bucket: position_tier. Outcome: ctr, with n. Mean AND median, since CTR is heavy-tailed.
"""

In [ ]:
order2 = ["top_3", "page_1", "striking", "page_3_5", "deep"]
position_table = df.groupby("position_tier")["ctr"].agg(
    mean_ctr="mean", median_ctr="median", n="count",
    zero_share=lambda s: (s == 0).mean()
).reindex(order2)
print(position_table.round(3))

In [ ]:
"""
Verdict: MIXED — real signal, but not where I first assumed.

By the mean, CTR falls cleanly as position worsens: top_3=1.484 down to deep=0.150, which
looks like a clean CONFIRMED story. But top_3 is 76.8% zero-CTR rows (n=2,321) — that mean
is being dragged up by a handful of extreme outliers, not a typical top_3 page. The median
tells a different, more honest story: page_1=0.16 is actually the highest stable median,
ahead of top_3=0.00. From page_1 downward, though, the median falls cleanly and
monotonically: page_1 0.16 -> striking 0.11 -> page_3_5 0.03 -> deep 0.00 — that part of the
position-to-CTR relationship holds up well. So: real signal, but I'm using page_1 as the
"ranks well" anchor for the rule below, not top_3, because top_3 is too sparse and
outlier-driven to trust in this snapshot.

"""

In [ ]:
"""
### The rule, in plain words

A page is worth a CTR-fix review if it already ranks on page 1 (position_tier == "page_1",
the reliable anchor from Signal check 2) but its click-through rate is below the typical
page-1 rate (under the page-1 median of 0.16), AND it still draws real search volume (not in
the lowest impression tier) — meaning the ranking effort already worked, but the
title/snippet isn't converting that visibility into clicks. This does not use staleness as a
driver, per Signal check 1's honest negative.

Reason code (one, for every flagged row): ctr_below_page1_median_with_volume

Action label: flag_for_ctr_fix (else no_action)

"""

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os

# Transparent score: readable conditions, no fitted weights (per building-baselines skill)
eligible_position = df["position_tier"].eq("page_1")
page1_median_ctr = df.loc[eligible_position, "ctr"].median()
underperforming_ctr = df["ctr"] < page1_median_ctr
has_real_volume = df["impression_tier"].ne("low")

flagged = eligible_position & underperforming_ctr & has_real_volume

df["reason_code"] = np.where(flagged, "ctr_below_page1_median_with_volume", "not_flagged")
df["action"] = np.where(flagged, "flag_for_ctr_fix", "no_action")
# Score = impressions_90d for flagged rows only, readable on purpose:
# bigger visible traffic already earned = bigger opportunity if CTR gets fixed
df["score"] = np.where(flagged, df["impressions_90d"], 0)

print("Benchmark (page_1 median CTR):", page1_median_ctr)
print("Eligible (page_1):", eligible_position.sum())
print("Flagged (eligible + underperforming CTR + real volume):", flagged.sum())

queue_cols = ["content_id", "client_id", "score", "reason_code", "action",
              "position_tier", "avg_position", "ctr", "impression_tier", "impressions_90d"]
queue = df[queue_cols].sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("../outputs", exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)
print("\nWrote", len(queue), "rows to work/outputs/baseline_action_score.csv")
queue.head(10)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = queue.head(20).copy()

def confidence_note(row):
    if row["ctr"] == 0:
        return f"High — zero recorded clicks despite page-1 ranking and {int(row['impressions_90d']):,} impressions."
    gap_pct = round((page1_median_ctr - row["ctr"]) / page1_median_ctr * 100, 1)
    return f"Moderate — CTR is {gap_pct}% below the page-1 median on {int(row['impressions_90d']):,} impressions."

def what_would_make_it_wrong(row):
    gap_pct = (page1_median_ctr - row["ctr"]) / page1_median_ctr
    if row["impressions_90d"] > df["impressions_90d"].quantile(0.99):
        return "If this impression count is a scrape/measurement outlier rather than real traffic, the score is inflated — worth a second data pull to confirm."
    if row["ctr"] == 0 and gap_pct >= 1.0:
        return "If a SERP feature (featured snippet, PAA, ads) is capturing the click above this result, a title/meta rewrite won't fix it — the opportunity may not be recoverable by content work alone."
    if gap_pct < 0.3:
        return "Gap to the page-1 median is small — within plausible day-to-day noise, so this could be a false positive, not a real underperformer."
    return "If the low CTR is seasonal or a one-off dip rather than a persistent pattern, fixing the title/meta won't move it — worth checking week-over-week stability first."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(what_would_make_it_wrong, axis=1)

for i, row in top20.iterrows():
    print(f"{i+1:2d}. content_id={row['content_id']}  action={row['action']}  reason={row['reason_code']}")
    print(f"    confidence: {row['confidence_note']}")
    print(f"    would be wrong if: {row['what_would_make_it_wrong']}")
    print()

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Weak picks: flag any top-20 row that's marginal rather than a clean case.
# Note: NOT using "impressions_90d is an outlier" as a weak-pick test here — the queue is
# ranked BY impressions_90d, so the top 20 always exceed any high percentile of it by
# construction. That check is still a valid per-row caveat in Section 3, but as a top-20-wide
# filter it would trivially mark all 20 "weak," which isn't a real finding.
# The genuinely discriminating signal is how CLOSE to the page-1 median a pick's CTR is —
# a small gap means the "underperformance" is marginal, possibly noise, not a clean case.
top20["ctr_gap_pct"] = (page1_median_ctr - top20["ctr"]) / page1_median_ctr
weak = top20[top20["ctr_gap_pct"] < 0.3]
print("Weak picks in the top 20 (CTR gap under 30% of the page-1 median — marginal, not clean):",
      len(weak), "out of 20")
if len(weak):
    print(weak[["content_id", "ctr", "ctr_gap_pct", "impressions_90d"]].round(3))
else:
    print("None found by this rule — worth a second look since building-baselines says finding zero weak picks means looking harder.")

# Leakage check: confirm the score/rule never touched label-derived or product columns
LABEL_DERIVED = ["trend_direction", "trend_pct",
                  "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                  "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
PRODUCT_FLAGS = ["provider_used", "model_used"]

RULE_INPUTS = ["position_tier", "ctr", "impression_tier", "impressions_90d"]
leaked_label = [c for c in RULE_INPUTS if c in LABEL_DERIVED]
leaked_product = [c for c in RULE_INPUTS if c in PRODUCT_FLAGS]

print("\nRule inputs used:", RULE_INPUTS)
print("Any label-derived columns among rule inputs?", leaked_label or "None — clean")
print("Any product/production-detail flags among rule inputs?", leaked_product or "None — clean")

future_like = [c for c in queue.columns if "future" in c.lower() or "next" in c.lower()]
print("Any future-window columns in the written queue?", future_like or "None — clean")

print("\nColumns actually written to the CSV:", list(queue.columns))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.